In [44]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
# tensor: fundamental building block

# direct creation from data
data = [[1, 2, 3], [4, 5, 6]]
my_tensor = torch.tensor(data)

print(my_tensor)

tensor([[1, 2, 3],
        [4, 5, 6]])


In [ ]:
# creation from a desired shape
# useful when the shape is known, but the values are not 

shape = (2, 3)

ones = torch.ones(shape)
zeros = torch.zeros(shape)
random = torch.randn(shape)

print(random)

tensor([[-1.3872,  0.3676, -0.2538],
        [ 0.5073,  0.5208, -1.3946]])


In [6]:
# creation by mimicking another tensor
# useful when a tensor with the same shape and type as another one is needed

template = torch.tensor([[1, 2], [3, 4]])

rand_like = torch.randn_like(template, dtype=torch.float)

print(f'Template tensor:\n {template}\n')
print(f'Randn_like tensor:\n {rand_like}\n')

Template tensor:
 tensor([[1, 2],
        [3, 4]])

Randn_like tensor:
 tensor([[ 0.1776, -0.8930],
        [-1.3354,  0.0304]])



In [ ]:
# inside a tensor: shape, type and device

tensor = torch.randn(shape)

print(f'Shape: {tensor.shape}')  # touple describing the dimensions
print(f'Datatype: {tensor.dtype}')  # data type, default is float32
print(f'Device: {tensor.device}')  # where the tensor lives: cpu or gpu

Shape: torch.Size([2, 3])
Datatype: torch.float32
Device: cpu


In [8]:
# by default, a tensor is just data
# to tell Pytorch that a tensor is a learnable parameter: requires_grad = True

x_data = torch.tensor([[1., 2.], [3., 4,]])  # standard data tensor
w = torch.tensor([[1.0], [2.0]], requires_grad=True)  # parameter tensor

print(f'Data tensor requires_grad: {x_data.requires_grad}')
print(f'Parameter tensor requires_grad: {w.requires_grad}')

Data tensor requires_grad: False
Parameter tensor requires_grad: True


In [9]:
# building a graph through operations

a = torch.tensor(2., requires_grad=True)
b = torch.tensor(3., requires_grad=True)
x = torch.tensor(4., requires_grad=True)

y = a + b
z = x * y

print(z)

tensor(20., grad_fn=<MulBackward0>)


In [ ]:
# every tensor created by an operation has the special attribute .grad_fn
# Pytorch builds a graph by registering the operations through this attribute

print(a.grad_fn)  # created by the user, not by an operation
print(y.grad_fn)  # created by addition
print(z.grad_fn)  # created by multiplication

None


In [15]:
# operations

a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[10, 20], [30, 40]])
print(f'a = {a}\n')
print(f'b = {b}\n')

# * multiplies matching positions (the tensors must have the same shape)
print(f'a * b = {a * b}\n')

# @ matrix multiplication (number of cols of 1st matrix must match rows of 2nd matrix)
print(f'a @ b = {a @ b}')

a = tensor([[1, 2],
        [3, 4]])

b = tensor([[10, 20],
        [30, 40]])

a * b = tensor([[ 10,  40],
        [ 90, 160]])

a @ b = tensor([[ 70, 100],
        [150, 220]])


**When building a linear layer, always use the `@` operator.**

In [16]:
# reduction operations: reduce a tensor to a smaller number of elements
# e.g. sum, mean, max

scores = torch.tensor([[10., 20., 30.], [5., 10., 15.]])

print(f'Overall mean: {scores.mean()}')

Overall mean: 15.0


In [18]:
# the dim argument lets us control which direction to collapse
# for 2D tensors:
# dim = 0 collapses the rows (operates vertically)
# dim = 1 collapses the columns (operates horizontally)

print(f'Col means: {scores.mean(dim=0)}')
print(f'Row means: {scores.mean(dim=1)}')

Col means: tensor([ 7.5000, 15.0000, 22.5000])
Row means: tensor([20., 10.])


In [20]:
# indexing (same as numpy)

x = torch.arange(12).reshape(3, 4)
col_2 = x[:, 2]

print(f'x = {x}')
print(f'Column 2: {col_2}')


x = tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Column 2: tensor([ 2,  6, 10])


In [22]:
# argmax: finds the index of the highest value
# this is how we find a model's final prediction

scores = torch.tensor([
    [10., 0., 5., 20., 1.],
    [1., 30., 2., 5., 0.]])

best_indices = torch.argmax(scores, dim=1)

print(f'Best indices for each row: {best_indices}')

Best indices for each row: tensor([3, 1])


In [23]:
# torch.gather: specific indexing

data = torch.tensor([
    [10, 11, 12, 13],
    [20, 21, 22, 23],
    [30, 31, 32, 33]
    ])

# which column to get from each row?
indices_to_select = torch.tensor([[2], [0], [3]])

# gather from data along dim=1 (columns)
selected_values = torch.gather(data, dim=1, index=indices_to_select)

print(selected_values)

tensor([[12],
        [20],
        [33]])


## Full example

In [33]:
# step 1: forward pass (first guess)
# we choose a simple linear regression model: y_hat = XW + b
# objective: find the best values for X and b

# setup: create data
N = 10  # number of points
# each data point has 1 input feature and 1 output value
D_in = 1
D_out = 1

# create input data X
X = torch.randn(N, D_in)

# create our true target labels y by using the "true" W and b
# the "true" W is 2.0 and the "true" b is 1.0
true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_W + true_b + torch.randn(N, D_out) * .1  # last term is noise

The job of the algorithm is to discover the parameters (true_W and true_b) just by looking at X and y_true.

In [34]:
# initialize the model's parameters (with random values)
# shapes must be correct for matrix multiplication
# requires_grad = True to specify that these are parameters

W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f'Initial weight W: {W}')
print(f'Initial bias b: {b}')

Initial weight W: tensor([[1.0285]], requires_grad=True)
Initial bias b: tensor([-0.0519], requires_grad=True)


In [35]:
y_hat = X @ W + b
print(f"Shape of our prediction y_hat: {y_hat.shape}\n")
print(f"Prediction y_hat (first 3 rows):\n {y_hat[:3]}\n")
print(f"True Labels y_true (first 3 rows):\n {y_true[:3]}")

Shape of our prediction y_hat: torch.Size([10, 1])

Prediction y_hat (first 3 rows):
 tensor([[-1.8582],
        [-0.5324],
        [-1.1375]], grad_fn=<SliceBackward0>)

True Labels y_true (first 3 rows):
 tensor([[-2.5032],
        [ 0.1704],
        [-1.0315]])


In [36]:
# loss: score that tells how badly the model is doing
# mean squared error (mse) is a type of loss

error = y_hat - y_true
squared_error = error**2
loss = squared_error.mean()  # has grad_fn too
print(f'Loss = {loss}')

Loss = 2.37738037109375


In [37]:
# calculate the gradient of the loss with respect to W and b
loss.backward()

print(f'Gradient for W (dL/dW): {W.grad}')
print(f'Gradient for b (dL/db): {b.grad}')

Gradient for W (dL/dW): tensor([[-2.6814]])
Gradient for b (dL/db): tensor([-2.0454])


Positive gradient -> Decreasing the parameter decreases the loss

Negative gradient -> Increasing the parameter decreases the loss

#### The Algorithm: Gradient Descent

The core update rule for gradient descent:

`θ_t+1 = θ_t - η * ∇_θ L`

*   `θ`: Parameters (`W` and `b` in this example).
*   `η` (eta): **learning rate**, a small number that controls how big of a step we take.
*   `∇_θ L`: gradient of the loss with respect to the parameters (`W.grad` and `b.grad`).

So, the update rules for our model are:
1.  `W_new = W_old - learning_rate * W.grad`
2.  `b_new = b_old - learning_rate * b.grad`

In [39]:
# training loop: repeat the 5 steps for multiple epochs
# torch.no_grad(): don't track parameter updates
# .grad.zero(): reset gradients each iteration

# Hyperparameters
learning_rate = 0.01  # eta
epochs = 200  # iterations

# re-initialize parameters with random values
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Starting Parameters: W={W.item():.3f}, b={b.item():.3f}\n")

# The Training Loop
for epoch in range(epochs):
    ### STEP 1 & 2: Forward Pass (guess) and Loss Calculation ###
    y_hat = X @ W + b
    loss = torch.mean((y_hat - y_true)**2)

    ### STEP 3: Backward Pass (Calculate Gradients) ###
    loss.backward()

    ### STEP 4: Update Parameters (The Gradient Descent Step) ###
    # We wrap this in no_grad() because this is not part of the model's computation
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    ### STEP 5: Zero the Gradients ###
    # We must reset the gradients for the next iteration
    W.grad.zero_()
    b.grad.zero_()

    # Optional: Print progress
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

print(f"\nFinal Parameters: W={W.item():.3f}, b={b.item():.3f}")
print(f"True Parameters:  W=2.000, b=1.000")

Starting Parameters: W=-0.184, b=-0.278

Epoch 00: Loss=8.3122, W=-0.121, b=-0.255
Epoch 10: Loss=4.8213, W=0.414, b=-0.041
Epoch 20: Loss=2.8263, W=0.811, b=0.139
Epoch 30: Loss=1.6759, W=1.106, b=0.290
Epoch 40: Loss=1.0062, W=1.325, b=0.417
Epoch 50: Loss=0.6122, W=1.488, b=0.523
Epoch 60: Loss=0.3779, W=1.609, b=0.611
Epoch 70: Loss=0.2369, W=1.699, b=0.684
Epoch 80: Loss=0.1512, W=1.766, b=0.745
Epoch 90: Loss=0.0985, W=1.817, b=0.795
Epoch 100: Loss=0.0658, W=1.854, b=0.837
Epoch 110: Loss=0.0452, W=1.882, b=0.871
Epoch 120: Loss=0.0322, W=1.903, b=0.900
Epoch 130: Loss=0.0239, W=1.919, b=0.923
Epoch 140: Loss=0.0185, W=1.931, b=0.943
Epoch 150: Loss=0.0150, W=1.940, b=0.959
Epoch 160: Loss=0.0128, W=1.947, b=0.972
Epoch 170: Loss=0.0113, W=1.952, b=0.982
Epoch 180: Loss=0.0103, W=1.956, b=0.991
Epoch 190: Loss=0.0097, W=1.959, b=0.999

Final Parameters: W=1.961, b=1.004
True Parameters:  W=2.000, b=1.000


## What if the model has a lot more layers and parameters?

The `torch.nn.Linear` layer does exactly what our manual `X @ W + b` operation did. It's a container that holds the `W` and `b` tensors for a linear transformation and performs the operation for us.

In [40]:
# The input to our model has 1 feature (D_in=1)
# The output of our model is 1 value (D_out=1)
D_in = 1
D_out = 1

# Create a Linear layer
linear_layer = torch.nn.Linear(in_features=D_in, out_features=D_out)

# You can inspect the randomly initialized parameters inside
print(f"Layer's Weight (W): {linear_layer.weight}\n")
print(f"Layer's Bias (b): {linear_layer.bias}\n")

# You use it just like a function. Let's pass our data X through it.
# This performs the forward pass: X @ W.T + b
# (Note: nn.Linear stores W as (D_out, D_in), so it uses a transpose)
y_hat_nn = linear_layer(X)

print(f"Output of nn.Linear (first 3 rows):\n {y_hat_nn[:3]}")

Layer's Weight (W): Parameter containing:
tensor([[0.7519]], requires_grad=True)

Layer's Bias (b): Parameter containing:
tensor([-0.4472], requires_grad=True)

Output of nn.Linear (first 3 rows):
 tensor([[-1.7677],
        [-0.7985],
        [-1.2409]], grad_fn=<SliceBackward0>)


## Neural network example

### Linear, one layer

In [46]:
# shape
N = 100
M = 10

# generate data
X = torch.randn(N, M)
y = torch.randn(N, 1)

# Model
model = nn.Linear(M, 1)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)  # parameters and learning rate

# Training loop
for epoch in range(1000):

    # STEP 1: Forward pass
    y_pred = model(X)

    # STEP 2: Compute loss
    loss = criterion(y_pred, y)

    # STEP 3: Backpropagation
    optimizer.zero_grad()  # clears the gradients from the previous training iteration
    loss.backward()  # gradient calculation
    optimizer.step()  # moves the weights in the direction that should reduce the loss

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.4f}")

print("Final output shape:", model(X).shape)

Epoch 0, Loss = 1.5447
Epoch 100, Loss = 0.8197
Epoch 200, Loss = 0.7957
Epoch 300, Loss = 0.7939
Epoch 400, Loss = 0.7937
Epoch 500, Loss = 0.7937
Epoch 600, Loss = 0.7937
Epoch 700, Loss = 0.7937
Epoch 800, Loss = 0.7937
Epoch 900, Loss = 0.7937
Final output shape: torch.Size([100, 1])


### Multiple linear layers

In [47]:
# -------------------------
# Data
# -------------------------

N = 100
M = 10

X = torch.randn(N, M)   # (100, 10)
y = torch.randn(N, 1)   # (100, 1)


# -------------------------
# Model
# -------------------------

model = nn.Sequential(
    nn.Linear(10, 20),
    nn.Linear(20, 10),
    nn.Linear(10, 1)
)


# -------------------------
# Loss and optimizer
# -------------------------

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


# -------------------------
# Training
# -------------------------

for epoch in range(1000):

    # Forward pass
    y_pred = model(X)

    # Calculate loss
    loss = criterion(y_pred, y)

    # Clear old gradients
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update weights
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")


print("Input shape: ", X.shape)
print("Output shape:", model(X).shape)

Epoch 0: loss = 0.9433
Epoch 100: loss = 0.8869
Epoch 200: loss = 0.8812
Epoch 300: loss = 0.8786
Epoch 400: loss = 0.8774
Epoch 500: loss = 0.8767
Epoch 600: loss = 0.8764
Epoch 700: loss = 0.8762
Epoch 800: loss = 0.8761
Epoch 900: loss = 0.8761
Input shape:  torch.Size([100, 10])
Output shape: torch.Size([100, 1])


**Important**: Despite having multiple layers, this network is still mathematically just one linear transformation. Stacking linear layers without anything between them doesn't make the network more expressive than a single linear layer. To change this, put a nonlinear activation function between the linear layers.